# G2 EnergyGuide data-quality profile

## tl;dr

Hosted run 35555806408 collected a parsed EnergyGuide source for all 75 exact refrigerator SKUs. The corpus contains 58 byte-distinct PDFs. Annual-energy review binding is complete for 8 SKUs/4 PDF hashes and capacity binding for 9 SKUs/5 hashes. The remaining values stay `NOT_OBSERVED`; overall product compliance stays `NOT_EVALUATED`.

## Context & Methods

The intended grain is one EnergyGuide fact per exact SKU. Manual review can be reused only through a byte-identical `pdf_sha256`, while each exact SKU remains independently represented. The notebook reads the immutable Actions artifact and invokes the repository quality profiler.

### Key Assumptions

- The downloaded ZIP is artifact `g2-pilot-35555806408-1` from successful hosted run 35555806408.
- A parsed PDF is source completeness, not an approved field value or compliance conclusion.
- Shared hashes reduce review workload but do not merge exact-SKU identities.

In [1]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'scripts'))
from g2_label_quality_report import build_quality_profile

artifact_path = Path(os.environ.get(
    'RDA_G2_ARTIFACT_ZIP',
    repo_root / 'runtime' / 'g2-pilot-35555806408.zip',
))
profile = build_quality_profile(artifact_path, github_run_id='35555806408')

## Data & Results

In [2]:
summary_keys = [
    'population_exact_skus', 'energyguide_facts', 'source_pdf_parsed',
    'unique_pdf_hashes', 'raw_model_values', 'raw_model_not_observed',
    'annual_energy_values', 'annual_energy_not_observed',
    'capacity_values', 'capacity_not_observed',
    'annual_energy_unreviewed_pdf_hashes', 'capacity_unreviewed_pdf_hashes',
]
[(key, profile['counts'][key]) for key in summary_keys]

[('population_exact_skus', 75), ('energyguide_facts', 75), ('source_pdf_parsed', 75), ('unique_pdf_hashes', 58), ('raw_model_values', 69), ('raw_model_not_observed', 6), ('annual_energy_values', 8), ('annual_energy_not_observed', 67), ('capacity_values', 9), ('capacity_not_observed', 66), ('annual_energy_unreviewed_pdf_hashes', 54), ('capacity_unreviewed_pdf_hashes', 53)]

In [3]:
[
    (row['exact_sku'], row['pdf_sha256'], row['fallback_reason'])
    for row in profile['records']
    if row['raw_model_state'] != 'VALUE'
]

[('RF90F23AECEAA', 'f8b7d79784f9fd8c87627a63c6a6a34474c02afca5e7cb1046d038b4f3120177', 'EMPTY_EMBEDDED_TEXT'), ('RF90F23AECRAA', 'f8b7d79784f9fd8c87627a63c6a6a34474c02afca5e7cb1046d038b4f3120177', 'EMPTY_EMBEDDED_TEXT'), ('RF90F23AEWAA', 'f8b7d79784f9fd8c87627a63c6a6a34474c02afca5e7cb1046d038b4f3120177', 'EMPTY_EMBEDDED_TEXT'), ('RF90F29AECEAA', '516173ddebf32cc3b67c64d12e04ce894165574896a081c9425ae9e332656e0a', 'EMPTY_EMBEDDED_TEXT'), ('RF90F29AECRAA', '516173ddebf32cc3b67c64d12e04ce894165574896a081c9425ae9e332656e0a', 'EMPTY_EMBEDDED_TEXT'), ('RF90F29AEWAA', '516173ddebf32cc3b67c64d12e04ce894165574896a081c9425ae9e332656e0a', 'EMPTY_EMBEDDED_TEXT')]

## Takeaways

1. Collection completeness is 75/75, with no missing EnergyGuide fact.
2. Review coverage, rather than download coverage, is the current blocker: 54 annual-energy PDF hashes and 53 capacity PDF hashes remain unreviewed.
3. RapidOCR is used for 71/75 SKUs, so raw text, coordinates, PDF hashes and panel review must remain together.
4. Six RF90F SKUs share two PDFs with multiple raw model patterns. Their model identity must remain unresolved until the pattern rule is reviewed.